# 6. Qualitative data (HAP-E)

Pull example sentences that fire each GoEmotions label, from `goemotions_sentence_probs.parquet`
(one row per sentence: `sentence` text + 28 `prob_<emotion>` columns + HAP-E metadata
`doc_id` / `author` / `genre` / `base_id`). A sentence fires an emotion when
`prob_<emo> >= THRESHOLDS[emo]` (the per-label tuned cuts, matching `4b`).

Then **section 7** ranks the parallel samples (`base_id`) by the sharpest emotional contrast
between `human` and the pooled `machine` (all 12 LLMs), and writes the qualitative data that
feeds the website: full reconstructed passage text per author for the most contrasting samples.
The per-sentence emotion tags for one sample are exported by `6a.export_sentence_sentiment.py`.

Token-level attribution is left as a marked stub (section 6).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

SENT_PATH = 'data_processed/goemotions_sentence_probs.parquet'
if not Path(SENT_PATH).exists():
    raise FileNotFoundError(f'{SENT_PATH} not found (cwd={Path.cwd()}).')

DROP_EMOTIONS = {'neutral'}   # fires on most sentences; excluded from the analysis

sp = pd.read_parquet(SENT_PATH)

# Require the HAP-E sentence schema (written by 4a). The pre-HAP-E LEAF cache lacks
# genre/base_id/source_tag; fail with a clear pointer instead of a raw KeyError later.
_HAPE_COLS = ['doc_id', 'author', 'genre', 'base_id', 'source_tag', 'sent_idx', 'sentence']
_missing = [c for c in _HAPE_COLS if c not in sp.columns]
if _missing:
    raise RuntimeError(
        f'{SENT_PATH} is missing HAP-E column(s) {_missing}. This looks like a pre-HAP-E '
        f'cache (e.g. the old LEAF parquet). Regenerate it by running '
        f'4a.GoEmotions_analysis_GPU.ipynb on the HAP-E corpus, which writes '
        f'goemotions_sentence_probs.parquet with doc_id/author/genre/base_id/source_tag/'
        f'sent_idx/sentence + prob_<emotion> columns.')

PROB_COLS = [c for c in sp.columns
             if c.startswith('prob_') and c[len('prob_'):] not in DROP_EMOTIONS]
EMOTIONS  = [c[len('prob_'):] for c in PROB_COLS]

# Per-label tuned thresholds (cirimus/modernbert-large-go-emotions model card, "Optimal
# Results"); identical to 4b.goemotions_perlabel_threshold_counts.py. A sentence fires
# emotion e iff prob_e >= THRESHOLDS[e].
THRESHOLDS = {
    'admiration': 0.40, 'amusement': 0.45, 'anger': 0.25, 'annoyance': 0.30,
    'approval': 0.30, 'caring': 0.35, 'confusion': 0.30, 'curiosity': 0.40,
    'desire': 0.40, 'disappointment': 0.30, 'disapproval': 0.35, 'disgust': 0.25,
    'embarrassment': 0.35, 'excitement': 0.25, 'fear': 0.40, 'gratitude': 0.50,
    'grief': 0.35, 'joy': 0.50, 'love': 0.45, 'nervousness': 0.45,
    'optimism': 0.25, 'pride': 0.15, 'realization': 0.25, 'relief': 0.25,
    'remorse': 0.65, 'sadness': 0.30, 'surprise': 0.40, 'neutral': 0.40,
}
THR = np.array([THRESHOLDS[e] for e in EMOTIONS])   # aligned to EMOTIONS / PROB_COLS

EMO_POSITIVE = {'admiration', 'amusement', 'approval', 'caring', 'desire', 'excitement',
                'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief'}
EMO_NEGATIVE = {'anger', 'annoyance', 'disappointment', 'disapproval', 'disgust',
                'embarrassment', 'fear', 'grief', 'nervousness', 'remorse', 'sadness'}

def valence(emo):
    if emo in EMO_POSITIVE:
        return 'positive'
    if emo in EMO_NEGATIVE:
        return 'negative'
    return 'ambiguous'   # confusion / curiosity / realization / surprise

print(f'{len(sp):,} sentences | {len(EMOTIONS)} emotions (per-label thresholds)')
print('authors:', sp['author'].unique().tolist())
print('genres:', sorted(sp['genre'].dropna().unique()))

In [ ]:
# HAP-E authors. Edit AUTHORS to retarget the example-sentence sections (3-5) below.
AUTHOR_ORDER = [
    'human',
    'gpt-4o', 'gpt-4o-mini', 'gpt-5-mini',
    'llama-3-70b', 'llama-3-70b-instruct', 'llama-3-8b', 'llama-3-8b-instruct',
    'gemma-2-9b', 'gemma-2-9b-it', 'gemma-2-27b', 'gemma-2-27b-it',
    'claude-haiku-4-5',
]
LLM_AUTHORS = [a for a in AUTHOR_ORDER if a != 'human']

# Default: every author (example sentences span the whole corpus).
AUTHORS = AUTHOR_ORDER
# Alternatives: pooled machine only -> AUTHORS = LLM_AUTHORS
#               a single model       -> AUTHORS = ['gemma-2-9b-it']
#               human only           -> AUTHORS = ['human']

# Keep only the requested authors actually present in the parquet (the AUTHOR_ORDER list
# is the full intended roster; a given corpus build may carry a subset). Drop the rest with
# a note rather than failing -- matches the "absent authors silently skipped" behaviour in 4c.
present = set(sp['author'].unique())
absent  = [a for a in AUTHORS if a not in present]
if absent:
    print(f'note: {len(absent)} requested author(s) absent from the parquet, skipped: {absent}')
AUTHORS = [a for a in AUTHORS if a in present]
assert AUTHORS, 'none of the requested authors are present in the parquet'

sp_sel = sp[sp['author'].isin(AUTHORS)].copy()
print(f'selected {len(sp_sel):,} / {len(sp):,} sentences across {len(AUTHORS)} author(s)')
display(sp_sel['author'].value_counts().rename('n_sentences').to_frame())

In [ ]:
# Fire count per emotion within the selected authors, ranked inside each valence group.
# Firing uses the per-label thresholds THR (aligned to PROB_COLS), not a flat cut.
TOP_K = 3   # how many emotions per valence to surface

fires = (sp_sel[PROB_COLS].to_numpy() >= THR)
fire_counts = pd.Series(fires.sum(axis=0), index=EMOTIONS, name='n_fired')

rank = (fire_counts.rename_axis('emotion').reset_index()
        .assign(valence=lambda d: d['emotion'].map(valence),
                frac=lambda d: d['n_fired'] / len(sp_sel)))
rank = rank.sort_values(['valence', 'n_fired'], ascending=[True, False])

top_by_valence = {v: rank[rank['valence'] == v]['emotion'].head(TOP_K).tolist()
                  for v in ['positive', 'negative', 'ambiguous']}

display(rank.groupby('valence', group_keys=False).head(TOP_K)
            .reset_index(drop=True))
print(top_by_valence)

In [ ]:
def emotion_examples(emo, n=8, min_prob=None, authors=None,
                     min_words=4, dedup=True, frame=None):
    """Top-probability sentences firing `emo`. Returns sentence/prob/author/genre/base_id/sent_idx.

    `min_prob` defaults to the per-label threshold THRESHOLDS[emo]."""
    frame = sp_sel if frame is None else frame
    if min_prob is None:
        min_prob = THRESHOLDS[emo]
    col = f'prob_{emo}'
    d = frame if authors is None else frame[frame['author'].isin(authors)]
    hit = d[d[col] >= min_prob].copy()
    if min_words:
        hit = hit[hit['sentence'].str.split().str.len() >= min_words]
    if dedup:
        hit = hit.sort_values(col, ascending=False).drop_duplicates('sentence')
    hit = hit.sort_values(col, ascending=False).head(n)
    out = hit[['sentence', col, 'author', 'genre', 'base_id', 'sent_idx']].rename(columns={col: 'prob'})
    return out.reset_index(drop=True)

In [ ]:
N_EXAMPLES = 6
pd.set_option('display.max_colwidth', 200)

for val in ['positive', 'negative', 'ambiguous']:
    for emo in top_by_valence[val]:
        print(f'\n=== {val.upper()} :: {emo}  (fired {int(fire_counts[emo]):,} sentences) ===')
        display(emotion_examples(emo, n=N_EXAMPLES))

## 6. Token-level attribution (stub)

GoEmotions is sentence-level: it emits one probability vector per sentence, with no per-token
label. Surfacing which tokens drive an emotion requires a post-hoc explainability method
(integrated gradients / SHAP / attention) run back through the model — a separate, heavier
pass with approximate output. Scaffold left below; not implemented.

In [ ]:
# TOKEN ATTRIBUTION -- NOT IMPLEMENTED.
# Plan: reload the GoEmotions model (CUDA), run integrated gradients per (sentence, emotion)
# on the fired rows above, return per-token attribution scores aligned to the tokenizer.
def token_attributions(sentence, emo):
    raise NotImplementedError('token-level attribution stub; see section 6')

## 7. Most emotionally contrasting parallel samples (qualitative export)

Ranks the parallel samples (`base_id`, e.g. `acad_0001`) by the sharpest emotional contrast
between `human` and a target — pooled `machine` (all 12 LLMs) by default, or a **specific LLM**
via `CONTRAST_TARGET`. Per `base_id` and per emotion, each side's **fired-rate** is the fraction
of its sentences crossing the per-label threshold; the contrast score sums the absolute
human↔target difference across the focus emotions.

A **per-LLM divergence summary** then ranks the 12 LLMs by how far each diverges emotionally
from human (so the specific models are distinguishable). The top samples are written to a **wide
passage CSV** — one row per `base_id`, one column per author (full reconstructed text) — the
qualitative data the website draws on. Per-sentence tags for one sample come from
`6a.export_sentence_sentiment.py`.

In [ ]:
# Rank parallel samples (base_id) by human-vs-target emotional contrast.
HUMAN_AUTHOR    = 'human'
CONTRAST_TARGET = 'machine'            # 'machine' (all 12 LLMs pooled) or a specific LLM label
TARGET_AUTHORS  = LLM_AUTHORS if CONTRAST_TARGET == 'machine' else [CONTRAST_TARGET]
FOCUS_EMOTIONS  = EMOTIONS             # all non-neutral emotions; narrow this to focus
MIN_HUMAN_SENTS = 3                    # drop samples whose human passage is too short to rate

fb = sp[sp['author'].isin([HUMAN_AUTHOR] + TARGET_AUTHORS)].copy()
for e in FOCUS_EMOTIONS:
    fb[f'fire_{e}'] = (fb[f'prob_{e}'] >= THRESHOLDS[e]).astype(int)
fire_cols = [f'fire_{e}' for e in FOCUS_EMOTIONS]

hum   = fb[fb['author'] == HUMAN_AUTHOR].groupby('base_id')
hum_r = hum[fire_cols].mean()
hum_n = hum.size().rename('n_human')

tgt   = fb[fb['author'].isin(TARGET_AUTHORS)].groupby('base_id')
tgt_r = tgt[fire_cols].mean()
tgt_n = tgt.size().rename('n_target')

# Keep samples with a long-enough human passage that also have target sentences.
bases = hum_n[hum_n >= MIN_HUMAN_SENTS].index.intersection(tgt_r.index)

d = (tgt_r.loc[bases] - hum_r.loc[bases])                   # signed target - human per emotion
d.columns = [f'd_{e}' for e in FOCUS_EMOTIONS]
contrast_score = d.abs().sum(axis=1).rename('contrast_score')
rank = (d.join(contrast_score).join(hum_n).join(tgt_n)
         .sort_values('contrast_score', ascending=False))

CHOSEN_BASE = rank.index[0]                                  # override here if desired
print(f"target = {CONTRAST_TARGET!r}  |  FOCUS_EMOTIONS = {len(FOCUS_EMOTIONS)} emotions")
print(f'{len(rank)} parallel samples scored  |  CHOSEN_BASE = {CHOSEN_BASE}')
display(rank.head(10))

In [ ]:
# Per-LLM emotional divergence from human (distinguishes the specific models).
# For each LLM: mean over shared base_ids of sum_e |fired-rate_LLM - fired-rate_human|,
# plus the single emotion that diverges most on average. Ranks which models are least/most
# human-like emotionally.
fb_all = sp[sp['author'].isin([HUMAN_AUTHOR] + LLM_AUTHORS)].copy()
for e in FOCUS_EMOTIONS:
    fb_all[f'fire_{e}'] = (fb_all[f'prob_{e}'] >= THRESHOLDS[e]).astype(int)
_fc = [f'fire_{e}' for e in FOCUS_EMOTIONS]

h_rate = fb_all[fb_all['author'] == HUMAN_AUTHOR].groupby('base_id')[_fc].mean()
h_n    = fb_all[fb_all['author'] == HUMAN_AUTHOR].groupby('base_id').size()
ok_bases = h_n[h_n >= MIN_HUMAN_SENTS].index

rows = []
for a in LLM_AUTHORS:
    a_rate = fb_all[fb_all['author'] == a].groupby('base_id')[_fc].mean()
    common = h_rate.index.intersection(a_rate.index).intersection(ok_bases)
    if len(common) == 0:
        continue
    diff = (a_rate.loc[common] - h_rate.loc[common])        # per base_id, per emotion
    per_emo = diff.abs().mean(axis=0)                       # mean |diff| per emotion
    top_e = per_emo.idxmax()[len('fire_'):]
    rows.append({'llm': a,
                 'mean_contrast': float(diff.abs().sum(axis=1).mean()),
                 'top_emotion': top_e,
                 'n_bases': int(len(common))})

llm_divergence = (pd.DataFrame(rows)
                  .sort_values('mean_contrast', ascending=False)
                  .reset_index(drop=True))
print('Per-LLM emotional divergence from human (higher = less human-like):')
display(llm_divergence)

In [ ]:
# Reconstruct each author's full passage for a base_id by joining its sentences in order.
def passage(base_id, author):
    s = sp[(sp['base_id'] == base_id) & (sp['author'] == author)].sort_values('sent_idx')
    return ' '.join(s['sentence'].astype(str).tolist())

# Wide passage CSV for the single CHOSEN_BASE: one column per author (full text).
wide = pd.DataFrame([{
    'base_id': CHOSEN_BASE,
    'contrast_score': float(rank.loc[CHOSEN_BASE, 'contrast_score']),
    **{a: passage(CHOSEN_BASE, a) for a in AUTHOR_ORDER},
}])

WIDE_PATH = f'data_processed/qual_emotion_wide_{CHOSEN_BASE}.csv'
wide.to_csv(WIDE_PATH, index=False)
print(f'wrote {WIDE_PATH}  ({wide.shape[0]} row x {wide.shape[1]} cols)')
display(wide.T)

In [ ]:
# Top-N most contrasting parallel samples, each row carrying the same passage columns
# (one full-text column per author) plus the contrast score. This is the qualitative
# data artifact the website draws on.
TOP_N = 30
top_bases = rank.head(TOP_N).index.tolist()

recs = []
for b in top_bases:
    rec = {'base_id': b, 'genre': b.split('_')[0],
           'contrast_score': float(rank.loc[b, 'contrast_score'])}
    rec.update({a: passage(b, a) for a in AUTHOR_ORDER})
    recs.append(rec)
top_wide = pd.DataFrame(recs)

TOP_WIDE_PATH = f'data_processed/qual_emotion_wide_top{TOP_N}.csv'
top_wide.to_csv(TOP_WIDE_PATH, index=False)
print(f'FOCUS_EMOTIONS = {len(FOCUS_EMOTIONS)} emotions  |  human vs machine (pooled)')
print(f'wrote {TOP_WIDE_PATH}  ({top_wide.shape[0]} rows x {top_wide.shape[1]} cols)')
# Preview key columns; the concrete B-side author (gemma-2-9b-it) is shown when present.
_preview = [c for c in ['base_id', 'genre', 'contrast_score', 'human', 'gemma-2-9b-it']
            if c in top_wide.columns]
display(top_wide[_preview])